# Financial RAG — Colab Setup

**First:** Runtime → Change runtime type → **T4 GPU** → Save

**Then run cells top to bottom. Each cell only needs to run once per session.**

---
Before running Step 4, add your secrets in Colab:
- Left sidebar → 🔑 Secrets icon
- Add `GROQ_API_KEY` = your Groq key (get free at console.groq.com)
- Add `EDGAR_EMAIL` = any email
- Toggle 'Notebook access' ON for both

In [ ]:
# ── Step 1: Mount Drive (persistent storage across sessions) ──────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/FinancialRAG'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted. Data folder:', DRIVE_DIR)

In [1]:
# ── Step 2: Clone repo ────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/mayankpuvvala/Financial_RAG.git'
REPO_DIR = '/content/Financial_RAG'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd /content/Financial_RAG
print('Working directory:', os.getcwd())

c:\content\Financial_RAG
Working directory: c:\content\Financial_RAG


Cloning into '/content/Financial_RAG'...
c:\Users\MAYANK PUVVALA\Downloads\jupyter\Financial_RAG\venv\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
# ── Step 3: Install dependencies ─────────────────────────────────────────
!pip install -q -r requirements.txt

# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


NVIDIA GeForce MX250, 2048 MiB


In [ ]:
# ── Step 4: Write .env from Colab Secrets (keys never touch the notebook) ─
from google.colab import userdata

groq_key    = userdata.get('GROQ_API_KEY')
edgar_email = userdata.get('EDGAR_EMAIL')

with open('.env', 'w') as f:
    f.write(f'groq_api={groq_key}\n')
    f.write(f'edgar_email={edgar_email}\n')

print('.env written (keys from Colab Secrets — not exposed in notebook)')

.env written (keys from Colab Secrets — not exposed in notebook)


In [ ]:
# ── Step 5: Symlink data/ → Drive (survives session resets) ───────────────
import os, shutil

DRIVE_DIR  = '/content/drive/MyDrive/FinancialRAG'
LOCAL_DATA = '/content/Financial_RAG/data'

# Remove local data/ dir if it exists (not a symlink)
if os.path.isdir(LOCAL_DATA) and not os.path.islink(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)

# Create symlink
if not os.path.islink(LOCAL_DATA):
    os.symlink(DRIVE_DIR, LOCAL_DATA)

# Ensure subdirs exist on Drive
for d in ['raw', 'parsed', 'chunks', 'qdrant', 'test_sets']:
    os.makedirs(f'{DRIVE_DIR}/{d}', exist_ok=True)

print('Symlink: data/ →', DRIVE_DIR)
print('Contents:', os.listdir(LOCAL_DATA))

Symlink: data/ → /content/drive/MyDrive/FinancialRAG


In [8]:
# ── Step 6: Run ingestion pipeline ───────────────────────────────────────
# First run: downloads 35 filings + embeds ~16k chunks on T4 GPU (~15 min)
# Subsequent runs: skips already-done steps automatically
!python run_ingestion.py

Traceback (most recent call last):
  File "c:\content\Financial_RAG\run_ingestion.py", line 132, in <module>
    main(skip_download=args.skip_download, skip_index=args.skip_index)
  File "c:\content\Financial_RAG\run_ingestion.py", line 44, in main
    settings.data_dir.mkdir(parents=True, exist_ok=True)
  File "C:\Users\MAYANK PUVVALA\AppData\Local\Programs\Python\Python310\lib\pathlib.py", line 1175, in mkdir
    self._accessor.mkdir(self, mode)
FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'C:\\content\\Financial_RAG\\data'


In [9]:
# ── Step 6b: If downloads already exist on Drive, skip download ───────────
!python run_ingestion.py --skip-download

Traceback (most recent call last):
  File "c:\content\Financial_RAG\run_ingestion.py", line 132, in <module>
    main(skip_download=args.skip_download, skip_index=args.skip_index)
  File "c:\content\Financial_RAG\run_ingestion.py", line 44, in main
    settings.data_dir.mkdir(parents=True, exist_ok=True)
  File "C:\Users\MAYANK PUVVALA\AppData\Local\Programs\Python\Python310\lib\pathlib.py", line 1175, in mkdir
    self._accessor.mkdir(self, mode)
FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'C:\\content\\Financial_RAG\\data'


In [10]:
# ── Step 7: Test queries ─────────────────────────────────────────────────
!python query.py "What was Apple revenue in FY2024?"

10:31:08 | DEBUG | Classified 'What was Apple revenue in FY2024?…' → single_doc | ['AAPL'] | [2024]
10:31:08 | INFO | Query type : single_doc
Tickers    : ['AAPL']
Years      : [2024]
Reasoning  : The query asks for a specific financial metric for one company in a specific year.
Traceback (most recent call last):
  File "c:\content\Financial_RAG\query.py", line 102, in <module>
    result   = ask(question)
  File "c:\content\Financial_RAG\query.py", line 51, in ask
    retrieved = retrieve(
  File "c:\content\Financial_RAG\retrieval\retriever.py", line 75, in retrieve
    collections = _target_collections(list(tickers), list(years))
  File "c:\content\Financial_RAG\retrieval\retriever.py", line 46, in _target_collections
    available = set(list_collections())
  File "c:\content\Financial_RAG\retrieval\vector_store.py", line 95, in list_collections
    return sorted(c.name for c in get_client().get_collections().collections)
  File "c:\content\Financial_RAG\retrieval\vector_store.py", 

In [ ]:
!python query.py "Compare Microsoft and Google R&D spending in 2024"

In [ ]:
!python query.py "How did JPMorgan net income trend from 2023 to 2025?"

In [ ]:
!python query.py "What is the current Bitcoin price?"

In [ ]:
# ── Step 8 (optional): Run FastAPI with public URL via ngrok ──────────────
!pip install -q pyngrok

import subprocess, time
from pyngrok import ngrok

# Start server
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'api.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
time.sleep(5)  # wait for startup + model warm-up

# Public URL
tunnel = ngrok.connect(8000)
print('Public URL:', tunnel.public_url)
print('Swagger  :', tunnel.public_url + '/docs')
print('Health   :', tunnel.public_url + '/health')
print()
print('Test query:')
print(f'  curl -X POST {tunnel.public_url}/query -H "Content-Type: application/json" -d \'{{"question": "What was Apple revenue in FY2024?"}}\' | python -m json.tool')

# Stream logs (Ctrl+C to stop)
try:
    for line in proc.stdout:
        print(line.decode(), end='')
except KeyboardInterrupt:
    proc.terminate(); ngrok.kill()

In [ ]:
# ── Utilities ─────────────────────────────────────────────────────────────
# Check what collections are indexed
from retrieval.vector_store import list_collections, get_collection_stats
cols = list_collections()
print(f'{len(cols)} collections indexed:')
for c in cols:
    s = get_collection_stats(c)
    print(f'  {c}: {s["points_count"]} chunks')

In [ ]:
# Check Drive storage usage
!du -sh /content/drive/MyDrive/FinancialRAG/*